# Bloch with gradient

## Imports

In [173]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.constants import mu_0
import spindata
from scipy.fft import fft, fftshift, fftfreq

## Parameters

In [174]:
B_0 = 9.4 # Field in teslas
Temp = 298 # Temperature in kelvins
Conc_I = 55.5 # Concentration of I species in M
mu_0 = 4*np.pi*1e-7
gamma = spindata.gamma("1H")
M0_thermal = spindata.magnetization("1H", 9.4, 298, 111)

T1 = 2  # Longitudinal relaxation time (s)
T2 = 1  # Transverse relaxation time (s)
omega_offset_I = 100
M0 = M0_thermal

grad_z = 100e-8   
tau_DDF = 0.06


In [175]:
# Parameters for geometry
sample_length = 0.5  # cm
num_slices = 1000  # Number of slices
slice_positions = np.linspace(0, sample_length, num_slices)  # Positions of slices (cm)

## Bloch equations defined

In [176]:
def bloch_equations(t, M, Spin, gradient_strength = 0):
    dM = np.zeros_like(M)
    # Calculate Bz for each slice based on its position
    Bz_grad = gradient_strength * slice_positions  # Slice-specific Bz values
    omega_slice = Spin.omega_offset + gamma * Bz_grad
    
    if Spin.DDF:
        Mx_avg = np.mean(M[0::3], axis = 0)
        My_avg = np.mean(M[1::3], axis = 0)
        Mz_avg = np.mean(M[2::3], axis = 0)
    
    for i in range(num_slices):
        Mx, My, Mz = M[3 * i : 3 * i + 3]
        
        dM[3 * i] = -omega_slice[i] * My - Mx / Spin.T2  # dMx/dt
        dM[3 * i + 1] = +omega_slice[i] * Mx - My / Spin.T2 # dMy/dt
        dM[3 * i + 2] = -(Mz - Spin.M0) / Spin.T1 # dMz/dt
        #print(omega_slice[i] )
        if Spin.DDF:

            DDF_x = My*Mz/(1/(gamma * mu_0)) - 2/3 * My*Mz_avg/(1/(gamma * mu_0)) - 1/3 * Mz*My_avg/(1/(gamma * mu_0))
            DDF_y = -Mx*Mz/(1/(gamma * mu_0)) + 2/3 * Mx*Mz_avg/(1/(gamma * mu_0)) + 1/3 * Mz*Mx_avg/(1/(gamma * mu_0))
            DDF_z = 1/3 * Mx*My_avg/(1/(gamma * mu_0)) - 1/3 * My*Mx_avg/(1/(gamma * mu_0))

            dM[3 * i] -= DDF_x
            dM[3 * i + 1] -= DDF_y
            dM[3 * i + 2] -= DDF_z

    return dM


In [177]:
def bloch_equations_inderectDDF(t, M, M_S, Spin_I, Spin_S, gradient_strength = 0):
    dM = np.zeros_like(M)
    # Calculate Bz for each slice based on its position
    Bz_grad = gradient_strength * slice_positions  # Slice-specific Bz values
    omega_slice = Spin_I.omega_offset + gamma * Bz_grad
    # M_Sx_avg = np.mean(M_S[0::3], axis = 0)
    # M_Sy_avg = np.mean(M_S[1::3], axis = 0)
    # M_Sz_avg = np.mean(M_S[2::3], axis = 0)

    # M_x_avg = np.mean(M[0::3], axis = 0)
    # M_y_avg = np.mean(M[1::3], axis = 0)
    # M_z_avg = np.mean(M[2::3], axis = 0)
    
    for i in range(num_slices):
        Mx, My, Mz = M[3 * i : 3 * i + 3]

        M_Sx, M_Sy, M_Sz = M_S[3 * i : 3 * i + 3]
        
        dM[3 * i] = -omega_slice[i] * My - Mx / Spin_I.T2  # dMx/dt
        dM[3 * i + 1] = +omega_slice[i] * Mx - My / Spin_I.T2 # dMy/dt
        dM[3 * i + 2] = -(Mz - Spin_I.M0) / Spin_I.T1 # dMz/dt
        #print(omega_slice[i] )
        if Spin_I.DDF:

            DDF_x = -My * M_Sz * gamma * mu_0 * 2/3 * np.exp(-t / Spin_S.T1) #- 2/3 * My*M_Sz_avg/(1/(gamma * mu_0)) - 1/3 * M_Sz*M_y_avg/(1/(gamma * mu_0))
            DDF_y = +Mx * M_Sz * gamma * mu_0 * 2/3 * np.exp(-t / Spin_S.T1)#+ 2/3 * Mx*M_Sz_avg/(1/(gamma * mu_0)) + 1/3 * M_Sz*M_x_avg/(1/(gamma * mu_0))
            DDF_z = 0#1/3 * M_Sx*M_Sy_avg/(1/(gamma * mu_0)) - 1/3 * M_Sy*M_Sx_avg/(1/(gamma * mu_0))

            dM[3 * i] -= DDF_x
            dM[3 * i + 1] -= DDF_y
            dM[3 * i + 2] -= DDF_z

    return dM


In [178]:
class Spin():
    def __init__(self, M0 = 0.03, omega_offset = 50, T1 = 20, T2 = 10, DDF = False):
        
        self.M0 = M0 #The number of spins in the substrate
        self.omega_offset = omega_offset
        self.T1 = T1   #in unit of ppm
        self.T2 = T2   #in unit of ppm
        self.DDF = DDF   #in unit of ppm

## Rotation of a vector defined

In [179]:
def rot_mat(u, theta):
    u =u / np.sqrt(np.sum(u**2))
    c = np.cos(theta)
    s = np.sin(theta)
    return np.array([[u[0]**2 * (1 - c) + c, u[0]*u[1]*(1 - c) - u[2]*s, u[0]*u[2]*(1 - c) + u[1]*s],
                        [u[0]*u[1]*(1 - c) + u[2]*s, u[1]**2 * (1 - c) + c, u[1]*u[2]*(1 - c) - u[0]*s],
                        [u[0]*u[2]*(1 - c) - u[1]*s, u[1]*u[2]*(1 - c) + u[0]*s, u[2]**2 * (1 - c) + c]])

def rotation(M, angle, axis):
    M_rot = np.zeros_like(M)
    for i in range(num_slices):
        Mx, My, Mz = M[3 * i : 3 * i + 3]
        M_new = rot_mat(axis, angle) @ np.array([Mx, My, Mz])
        M_rot[3 * i] = M_new[0]# new Mx
        M_rot[3 * i + 1] = M_new[1]# new My
        M_rot[3 * i + 2] = M_new[2]# new Mz
    return M_rot

## MSE sequence for DDF

### First 90 degree and free evolution

In [180]:
spin = Spin(M0 = M0_thermal, omega_offset = omega_offset_I, T1 = T1, T2 = T2, DDF = True)

In [ ]:
# Time span for the simulation
t_span = (0, 0.1)  # seconds
N_steps = int(t_span[1]*1000)
t_eval = np.linspace(t_span[0], t_span[1], N_steps)
d_helix = 2*np.pi/(gamma*grad_z*t_span[1])
print(d_helix)
# Initial conditions for each slice
initial_conditions_slices = np.zeros((3 * num_slices,))

for i in range(num_slices):
    initial_conditions_slices[3 * i + 2] = spin.M0  # Mz initial for each slice

# 90 degree pulse along x
initial_conditions_slices = rotation(initial_conditions_slices, np.pi/2, np.array([1, 0 , 0]))

# Solve the Bloch equations with the gradient
solution_with_gradient = solve_ivp(
    bloch_equations, 
    t_span, initial_conditions_slices, t_eval=t_eval, 
    method="DOP853", rtol=1e-10, atol=1e-13, 
    args=[spin, grad_z]
)

In [ ]:
# Extract transverse magnetization for several representative slices
representative_slices = np.arange(0, num_slices, num_slices//5)
Mx_slices = solution_with_gradient.y[0::3]
My_slices = solution_with_gradient.y[1::3]

# Plotting the transverse magnetization (Mx and My) for several slices with gradient
plt.figure(figsize=(8, 4))
for idx in representative_slices:
    plt.plot(t_eval, Mx_slices[idx], label=f"Slice {idx} ($M_x$)", linestyle='-')
    #plt.plot(t_eval, My_slices[idx], label=f"Slice {idx} ($M_y$)", linestyle='--')

plt.title("Transverse Magnetization with Gradient (Representative Slices)")
plt.xlabel("Time (s)")
plt.ylabel("Magnetization (a.u.)")
#plt.ylim(-5e-5,5e-5)
#plt.xlim(0,2)
plt.legend()
plt.grid()
plt.show()

In [ ]:
# Averaging
Mx_avg = np.mean(Mx_slices, axis = 0)
My_avg = np.mean(My_slices, axis = 0)

# Plotting the transverse magnetization (Mx and My) for several slices with gradient
plt.figure(figsize=(8, 4))
plt.plot(t_eval, Mx_avg, label = "$M_x$")
plt.plot(t_eval, My_avg, label = "$M_y$")
#plt.ylim(-5e-5,5e-5)
plt.title("Averaged Transverse Magnetization with Gradient")
plt.xlabel("Time (s)")
plt.ylabel("Averaged Magnetization (a.u.)")
plt.legend()
plt.grid()
plt.show()


### Second pulse and free evolution

In [184]:
# Final Mx values along the z-axis (spatial positions of slices)
Mx_final = solution_with_gradient.y[0::3, -1]  # Take Mx at the last time step for all slices
# Final Mx values along the z-axis (spatial positions of slices)
My_final = solution_with_gradient.y[1::3, -1]  # Take Mx at the last time step for all slices

In [185]:
# Time span for the simulation
t_span = (0, 1)  # seconds
N_steps = t_span[1]*1000
t_eval = np.linspace(t_span[0], t_span[1], N_steps)

# 180 degree pulse along x
initial_conditions_slices = rotation(solution_with_gradient.y[:, -1], np.pi*120/180, np.array([1, 0 , 0]))
 
spin.DDF = False
# Solve the Bloch equations with the gradient
solution_with_gradient_FID = solve_ivp(
    bloch_equations, 
    t_span, initial_conditions_slices, t_eval=t_eval, 
    method="DOP853", rtol=1e-10, atol=1e-13, 
    args=[spin, grad_z]
)

spin.DDF = True
solution_with_gradient_DDF_FID = solve_ivp(
    bloch_equations, 
    t_span, initial_conditions_slices, t_eval=t_eval, 
    method="DOP853", rtol=1e-10, atol=1e-13, 
    args=[spin,grad_z]
)

In [ ]:
Mz = initial_conditions_slices[2::3]

# Plotting the transverse magnetization (Mx and My) for several slices with gradient
plt.figure(figsize=(8, 4))
plt.plot(slice_positions, Mz[:], label = "$M_z$")
#plt.ylim(-5e-5,5e-5)
plt.title("Long Magnetization after second pulse")
plt.xlabel("Slice")
plt.ylabel("Magnetization (a.u.)")
plt.legend()
plt.grid()
plt.show()

In [ ]:
Mx = solution_with_gradient_FID.y[0::3][:,300]
My = solution_with_gradient_FID.y[1::3][:,300]

Mx_DDF = solution_with_gradient_DDF_FID.y[0::3][:,300]
My_DDF = solution_with_gradient_DDF_FID.y[1::3][:,300]

plt.rcParams.update({'font.size': 14})
fig, axs = plt.subplots(2, 1, figsize=(8, 8), sharex = True)

axs[1].set_xlabel("slice_positions, cm")
axs[0].set_ylabel("|Magnetization|")
axs[1].set_ylabel("|Magnetization|")


axs[0].set_title("No DDF",fontsize=14)
axs[0].set_ylim(-3e-2, 3e-2)
axs[0].plot(slice_positions, Mx, label = "$M_x$")
axs[0].plot(slice_positions, My, label = "$M_y$")
axs[0].legend()

axs[1].set_title("With DDF",fontsize=14)
axs[1].set_ylim(-3e-2, 3e-2)
axs[1].plot(slice_positions, Mx_DDF, label = "$M_x$")
axs[1].plot(slice_positions, My_DDF, label = "$M_y$")
axs[1].legend()


In [ ]:
# Averaging
Mx_avg = np.mean(solution_with_gradient_FID.y[0::3], axis = 0)
My_avg = np.mean(solution_with_gradient_FID.y[1::3], axis = 0)

Mx_avg_DDF = np.mean(solution_with_gradient_DDF_FID.y[0::3], axis = 0)
My_avg_DDF = np.mean(solution_with_gradient_DDF_FID.y[1::3], axis = 0)

plt.rcParams.update({'font.size': 14})
fig, axs = plt.subplots(2, 1, figsize=(8, 8), sharex = True)

axs[1].set_xlabel("time, s")
axs[0].set_ylabel("Magnetization")
axs[1].set_ylabel("Magnetization")


axs[0].set_title("No DDF",fontsize=14)
axs[0].set_ylim(-2e-2, 2e-2)
axs[0].plot(t_eval, Mx_avg, label = "$M_x$")
axs[0].plot(t_eval, My_avg, label = "$M_y$")
axs[0].legend()

axs[1].set_title("With DDF",fontsize=14)
axs[1].set_ylim(-2e-2, 2e-2)
axs[1].plot(t_eval, Mx_avg_DDF, label = "$M_x$")
axs[1].plot(t_eval, My_avg_DDF, label = "$M_y$")
axs[1].legend()


In [ ]:
# Final Mx values along the z-axis (spatial positions of slices)
Mx_final = solution_with_gradient.y[0::3, -1]  # Take Mx at the last time step for all slices

# Perform FFT to calculate k-space
k_space = fftshift(fft(Mx))  # Center the zero frequency component
k_values = fftshift(fftfreq(num_slices, d=sample_length / num_slices))  # k-space frequencies

# Perform FFT to calculate k-space
k_space_2  = fftshift(fft(Mx_DDF))  # Center the zero frequency component

# Plot k-space (magnitude of FFT)
plt.figure(figsize=(10, 6))
plt.xlim(-50,50)
plt.plot(k_values, np.abs(k_space), marker='o', label="|FFT($M_x$)| no DDF")
plt.plot(k_values, np.abs(k_space_2), marker='o', label="|FFT($M_x$)| with DDF")
plt.title("k-Space Representation of Transverse Magnetization")
plt.xlabel("$k_z$ (1/cm)")
plt.ylabel("Magnitude (a.u.)")
plt.grid()
plt.legend()
plt.show()

## CRAZED simulations

In [190]:
def CRAZED(grad_1, tau, grad_2):
    d_helix = 2*np.pi/(gamma*grad_1*tau)
    print(f"Gradient helix period is {d_helix} cm")

    # Initial conditions for each slice
    initial_conditions_slices = np.zeros((3 * num_slices,))
    for i in range(num_slices):
        initial_conditions_slices[3 * i + 2] = M0  # Mz initial for each slice

    # 90 degree pulse along x
    initial_conditions_slices = rotation(initial_conditions_slices, np.pi/2, np.array([1, 0 , 0]))

    # Time span 1st evolution
    t_span = (0, tau)  # seconds
    N_steps = int(t_span[1]*1000)
    t_eval = np.linspace(t_span[0], t_span[1], N_steps)
    # Solve the Bloch equations with the gradient
    first_evolution = solve_ivp(
        bloch_equations, 
        t_span, initial_conditions_slices, t_eval=t_eval, 
        method="DOP853", rtol=1e-10, atol=1e-13, 
        args=[spin, grad_1]
    )

    # 90 degree pulse along x
    initial_conditions_slices = rotation(first_evolution.y[:, -1], np.pi*120/180, np.array([1, 0 , 0]))

    # Time span for the simulation
    t_span = (0, 5*tau)  # seconds
    N_steps = int(t_span[1]*1000)
    t_eval = np.linspace(t_span[0], t_span[1], N_steps)
    second_evolution = solve_ivp(
        bloch_equations, 
        t_span, initial_conditions_slices, t_eval=t_eval, 
        method="DOP853", rtol=1e-10, atol=1e-13, 
        args=[spin, grad_2]
    )

    return second_evolution

In [ ]:
N_grads = 4
tau = 0.2

Mx_grads = np.zeros([int(5*tau*1000), N_grads])
My_grads = np.zeros([int(5*tau*1000), N_grads])

for idx in range(4):
    signal = CRAZED(grad_z, tau, idx*grad_z)
    # Averaging
    Mx_avg = np.mean(signal.y[0::3], axis = 0)
    My_avg = np.mean(signal.y[1::3], axis = 0)

    Mx_grads[:,idx] = Mx_avg
    My_grads[:,idx] = My_avg

In [ ]:
plt.rcParams.update({'font.size': 14})
fig, axs = plt.subplots(N_grads, 1, figsize=(8, 3*N_grads), sharex=True)

axs[N_grads-1].set_xlabel("time, s")
for i in range(N_grads):
    
    axs[i].set_ylabel("Magnetization")
    axs[i].set_title(f"Gradient ratio is {i}",fontsize=14)
    axs[i].set_ylim(-2e-2, 2e-2)
    axs[i].plot(signal.t, Mx_grads[:,i], label = "$M_x$")
    axs[i].plot(signal.t, My_grads[:,i], label = "$M_y$")
    axs[i].legend()

## Indirect detection with distant dipolar fields

### Parameters

In [ ]:
B_0 = 9.4 # Field in Teslas
gamma = spindata.gamma("1H")

## Analyte spin S
T1_S = 20  # Longitudinal relaxation time (s)
T2_S = 10  # Transverse relaxation time (s)
omega_offset_S = 20
Conc_S = 1e-1 # Concentration of S species in M
M0_S = spindata.magnetization("1H", 9.4, 298, Conc_S)
tau_DDF_S = 1/(M0_S * mu_0 * gamma)
print(f"Magnetization of S spins is {M0_S:.5f} A/m and DDF time is {tau_DDF_S:.5f} s")
spin_S = Spin(M0 = M0_S, omega_offset = omega_offset_S, T1 = T1_S, T2 = T2_S, DDF = False)

## Sensor spin I
T1_I = 20  # Longitudinal relaxation time (s)
T2_I = 10  # Transverse relaxation time (s)
omega_offset_I = 10
Conc_I = 1 # Concentration of I species in M
P0_I = spindata.polarization("1H", 9.4, 0.022)
M0_I = spindata.magnetization("1H", 9.4, 0.022, Conc_I)
tau_DDF_I = 1/(M0_I * mu_0 * gamma)
print(f"Polatization of I spins is {P0_I:.5f}")
print(f"Magnetization of I spins is {M0_I:.5f} A/m and DDF time is {tau_DDF_I:.5f} s")
spin_I = Spin(M0 = M0_I, omega_offset = omega_offset_I, T1 = T1_I, T2 = T2_I, DDF = True)


In [194]:
# Parameters for geometry
sample_length = 1  # cm
num_slices = 2000  # Number of slices
slice_positions = np.linspace(0, sample_length, num_slices)  # Positions of slices (cm)
grad_z = 2 # in G/cm
grad_z = grad_z * 1e-4 # Gauss to Tesla

grad_1, tau, grad_ratio = grad_z, 0.1, 1

### Numerical calculation with Bloch equations

In [ ]:
d_helix = 2*np.pi/(gamma*grad_1*tau)
print(f"Gradient helix period is {d_helix} cm")

# Initial conditions for each slice
initial_conditions_slices_S = np.zeros((3 * num_slices,))
initial_conditions_slices_I = np.zeros((3 * num_slices,))
for i in range(num_slices):
    initial_conditions_slices_I[3 * i + 2] = spin_I.M0  # Mz initial for each slice
    initial_conditions_slices_S[3 * i + 2] = spin_S.M0  # Mz initial for each slice


# # 90 degree pulse along x
# initial_conditions_slices_S = rotation(initial_conditions_slices_S, np.pi/2, np.array([1, 0 , 0]))

# # Time span 1st evolution
# t_span = (0, tau)  # seconds
# N_steps = int(t_span[1]*1000)
# t_eval = np.linspace(t_span[0], t_span[1], N_steps)
# # Solve the Bloch equations with the gradient
# first_evolution_S = solve_ivp(
#     bloch_equations, 
#     t_span, initial_conditions_slices_S, t_eval=t_eval, 
#     method="DOP853", rtol=1e-10, atol=1e-13, 
#     args = [spin_S, grad_1]
# )
# # 90 degree pulse along x
# initial_conditions_slices_S = rotation(first_evolution_S.y[:, -1], np.pi*90/180, np.array([1, 0 , 0]))
for i in range(num_slices):
    Bz_grad = grad_1 * slice_positions[i]  # Slice-specific Bz values
    omega_slice = omega_offset_S - gamma * Bz_grad
    initial_conditions_slices_S[3 * i] = 0  # Mx initial for each slice
    initial_conditions_slices_S[3 * i + 1] = 0  # My initial for each slice
    initial_conditions_slices_S[3 * i + 2] = M0_S*np.cos(omega_slice * tau)

In [196]:

initial_conditions_slices_I = rotation(initial_conditions_slices_I, np.pi*90/180, np.array([1, 0 , 0]))
spin_I.DDF = False
# Time span for the simulation
t_span = (0, 1*tau)  # seconds
N_steps = int(t_span[1]*1000)
t_eval = np.linspace(t_span[0], t_span[1], N_steps)
second_evolution = solve_ivp(
    bloch_equations_inderectDDF, 
    t_span, initial_conditions_slices_I, t_eval=t_eval, 
    method="DOP853", rtol=1e-10, atol=1e-13, 
    args=[initial_conditions_slices_S, spin_I, spin_S, grad_ratio*grad_1]
)
spin_I.DDF = True
# Time span for the simulation
t_span = (0, 200*tau)  # seconds
N_steps = int(t_span[1]*500)
t_eval = np.linspace(t_span[0], t_span[1], N_steps)
final_evolution = solve_ivp(
    bloch_equations_inderectDDF, 
    t_span, second_evolution.y[:, -1], t_eval=t_eval, 
    method="DOP853", rtol=1e-10, atol=1e-13, 
    args=[initial_conditions_slices_S, spin_I, spin_S, 0]
)

In [ ]:
# Averaging 1
Mx_I_avg = np.mean(second_evolution.y[0::3], axis = 0)
My_I_avg = np.mean(second_evolution.y[1::3], axis = 0)
# Averaging 2
Mx_I_avg_2 = np.mean(final_evolution.y[0::3], axis = 0)
My_I_avg_2 = np.mean(final_evolution.y[1::3], axis = 0)

plt.rcParams.update({'font.size': 14})
fig, axs = plt.subplots(3, 1, figsize=(8, 12))


axs[0].set_ylabel("Long Magnetization of S")
axs[1].set_ylabel("Trans Magnetization of I/M0_I")
axs[2].set_ylabel("Trans Magnetization of I/M0_S")

axs[0].set_xlabel("Slice position, cm")
axs[1].set_xlabel("time_grad, s")
axs[2].set_xlabel("time_2, s")

#axs[0].set_title("No DDF",fontsize=14)
axs[0].set_xlim(0, 0.1)
axs[0].plot(slice_positions, initial_conditions_slices_S[2::3], label = "$M_z$ of S spin after second 90")
#axs[0].legend()

#axs[1].set_title("With DDF",fontsize=14)
#axs[1].set_ylim(-2e-2, 2e-2)
axs[1].plot(second_evolution.t, Mx_I_avg/spin_I.M0, label = "$M_x$")
axs[1].plot(second_evolution.t, My_I_avg/spin_I.M0, label = "$M_y$")
axs[1].legend()


axs[2].plot(final_evolution.t, Mx_I_avg_2/spin_S.M0, label = "$M_x$")
axs[2].plot(final_evolution.t, My_I_avg_2/spin_S.M0, label = "$M_y$")
axs[2].legend()

### Analytical formula

In [198]:
def Mz_S(t2, G_tau_1, s, G, spin_S):
    return spin_S.M0 * np.exp(-G_tau_1 / spin_S.T2) * np.cos(spin_S.omega_offset * G_tau_1 - gamma * G * G_tau_1 * s) * np.exp(-t2 / spin_S.T1)

def Mp_I(t2, s, G, G_ratio, G_tau_2, spin_I, B_DDF):
    return spin_I.M0 * np.exp(-t2 / spin_I.T2) * np.exp(1j * (spin_I.omega_offset * t2  - gamma * B_DDF * t2 - gamma * G_ratio * G * s * G_tau_2))

In [ ]:
t2 = np.linspace(0,50,5000)

T = tau
G = grad_z
G_ratio = 1
s = slice_positions

signal_x = np.zeros_like(t2)
signal_y = np.zeros_like(t2)

for idt, t in enumerate(t2):
    DDF_S = mu_0 * 2 / 3 * Mz_S(t, T, s, G, spin_S)
    signal_x[idt] = np.mean(np.real(Mp_I(t, s, G, G_ratio, T, spin_I, DDF_S)))
    signal_y[idt] = np.mean(np.imag(Mp_I(t, s, G, G_ratio, T, spin_I, DDF_S)))

plt.rcParams.update({'font.size': 14})
fig, axs = plt.subplots(2, 1, figsize=(8, 12))


axs[0].set_ylabel(r"$|M_{Tr}^{I}(t_{2})|$ / $M_{0}^{S}$")
axs[0].set_xlabel(r"$t_{2}$, s")
axs[0].set_ylim(-3000,3000)
axs[0].set_xlim(0,t_span[1])
axs[0].set_title(f"Numerical",fontsize=14)
axs[0].plot(final_evolution.t, Mx_I_avg_2/spin_S.M0, label = "$M_x$")
axs[0].plot(final_evolution.t, My_I_avg_2/spin_S.M0, label = "$M_y$")

axs[1].set_ylabel(r"$|M_{Tr}^{I}(t_{2})|$ / $M_{0}^{S}$")
axs[1].set_xlabel(r"$t_{2}$, s")
axs[1].set_ylim(-3000,3000)
axs[1].set_xlim(0,t_span[1])
axs[1].set_title(f"Analytical",fontsize=14)
axs[1].plot(t2, signal_x/spin_S.M0, label = "$M_x$")
axs[1].plot(t2, signal_y/spin_S.M0, label = "$M_y$")
#plt.ylabel("Averaged Magnetization (a.u.)")
#plt.legend()
#plt.grid()
plt.show()

In [ ]:
Enh_max = mu_0 / 3 * gamma * spin_I.M0 * spin_I.T2 * np.exp(-T/spin_S.T2 - 1)

print("Theoretical signal enhancement in linear regime:")
print(Enh_max)

### More testing with analytical formula

In [ ]:
B_0 = 9.4 # Field in Teslas
gamma = spindata.gamma("1H")

## Analyte spin S
T1_S = 60  # Longitudinal relaxation time (s)
T2_S = 30  # Transverse relaxation time (s)
omega_offset_S = 2

## Sensor spin I
T1_I = 60  # Longitudinal relaxation time (s)
T2_I = 30  # Transverse relaxation time (s)
omega_offset_I = 10
Conc_I = 1 # Concentration of I species in M
P0_I = spindata.polarization("1H", 9.4, 0.022)
M0_I = spindata.magnetization("1H", 9.4, 0.022, Conc_I)
tau_DDF_I = 1/(M0_I * mu_0 * gamma)
print(f"Polatization of I spins is {P0_I:.5f}")
print(f"Magnetization of I spins is {M0_I:.5f} A/m and DDF time is {tau_DDF_I:.5f} s")
spin_I = Spin(M0 = M0_I, omega_offset = omega_offset_I, T1 = T1_I, T2 = T2_I, DDF = True)

# Parameters for geometry
sample_length = 1  # cm
num_slices = 20000  # Number of slices
slice_positions = np.linspace(0, sample_length, num_slices)  # Positions of slices (cm)


In [ ]:
grads = [1e-6, 1e-5, 1e-4]
grad_ratio = 1
t2 = np.linspace(0,50,5000)

T = 0.1

signal_x = np.zeros([2, len(grads), t2.shape[0]])
signal_y = np.zeros([2, len(grads), t2.shape[0]])

Conc_S = 1e-1 # Concentration of S species in M
M0_S = spindata.magnetization("1H", 9.4, 298, Conc_S)
tau_DDF_S = 1/(M0_S * mu_0 * gamma)
print(f"Magnetization of S spins is {M0_S:.5f} A/m and DDF time is {tau_DDF_S:.5f} s")
spin_S = Spin(M0 = M0_S, omega_offset = omega_offset_S, T1 = T1_S, T2 = T2_S, DDF = False)

for idg, g in enumerate(grads):
    for idt, t in enumerate(t2):
        DDF_S = mu_0 * 2 / 3 * Mz_S(t, T, s, g, spin_S)
        signal_x[0, idg, idt] = np.mean(np.real(Mp_I(t, s, g, G_ratio, T, spin_I, DDF_S)))/spin_S.M0
        signal_y[0, idg, idt] = np.mean(np.imag(Mp_I(t, s, g, G_ratio, T, spin_I, DDF_S)))/spin_S.M0

Conc_S = 1e-2 # Concentration of S species in M
M0_S = spindata.magnetization("1H", 9.4, 298, Conc_S)
tau_DDF_S = 1/(M0_S * mu_0 * gamma)
print(f"Magnetization of S spins is {M0_S:.5f} A/m and DDF time is {tau_DDF_S:.5f} s")
spin_S.M0 = M0_S

for idg, g in enumerate(grads):
    for idt, t in enumerate(t2):
        DDF_S = mu_0 * 2 / 3 * Mz_S(t, T, s, g, spin_S)
        signal_x[1, idg, idt] = np.mean(np.real(Mp_I(t, s, g, G_ratio, T, spin_I, DDF_S)))/spin_S.M0
        signal_y[1, idg, idt] = np.mean(np.imag(Mp_I(t, s, g, G_ratio, T, spin_I, DDF_S)))/spin_S.M0

In [ ]:
plt.rcParams.update({'font.size': 12})
fig = plt.figure(figsize=(6, 2*len(grads)), constrained_layout=True)
subfigs = fig.subfigures(1, 2)

# axs[len(grads)-1, 0].set_xlabel("time, s")
# axs[len(grads)-1, 1].set_xlabel("time, s")
titles = [f"[S] = 100 mM",f"[S] = 10 mM"]

for i, subfig in enumerate(subfigs):
    axs = subfig.subplots(len(grads), 1, sharex= True)
    for idg in range(len(grads)):
        if i==0: axs[idg].set_ylabel(r"$|M_{Tr}^{I}(t_{2})|$ / $M_{0}^{S}$")
        axs[idg].set_title(f"G = {grads[idg]*1e+6:.1f} uT/cm",fontsize=14)
        #axs[idg, 0].set_ylim(-4e+3, 4e+3)
        axs[idg].plot(t2, signal_x[i, idg, :], label = "$M_x$")
        axs[idg].plot(t2, signal_y[i, idg, :], label = "$M_y$")
        #axs[idg, 1].set_ylim(-3e+3, 3e+3)
    axs[len(grads)-1].set_xlabel("time, s")
    subfig.suptitle(titles[i])

fig.suptitle(f"DDF signal of Maleate at {P0_I*Conc_I:.2f} M polarization\n" + 
             r'Maleate has T$_{1}$='+f"{T1_I} s, " + r'T$_{2}$='+f"{T2_I} s\n" + 
             r"Substrate has T$_{1}$=" + f"{T1_S} s, " + r'T$_{2}$='+f"{T2_S} s\n"
             
              ,fontsize=16)

plt.show()


In [ ]:
grads = [1e-6, 1e-5, 1e-4]
grad_ratio = 1
t2 = np.linspace(0,50,5000)

T = 0.1

signal_x = np.zeros([2, len(grads), t2.shape[0]])
signal_y = np.zeros([2, len(grads), t2.shape[0]])

T1_S = 10
T2_S = 5

spin_I.T1 = T1_S
spin_I.T2 = T2_S

Conc_S = 1e-1 # Concentration of S species in M
M0_S = spindata.magnetization("1H", 9.4, 298, Conc_S)
tau_DDF_S = 1/(M0_S * mu_0 * gamma)
print(f"Magnetization of S spins is {M0_S:.5f} A/m and DDF time is {tau_DDF_S:.5f} s")
spin_S = Spin(M0 = M0_S, omega_offset = omega_offset_S, T1 = T1_S, T2 = T2_S, DDF = False)

for idg, g in enumerate(grads):
    for idt, t in enumerate(t2):
        DDF_S = mu_0 * 2 / 3 * Mz_S(t, T, s, g, spin_S)
        signal_x[0, idg, idt] = np.mean(np.real(Mp_I(t, s, g, G_ratio, T, spin_I, DDF_S)))/spin_S.M0
        signal_y[0, idg, idt] = np.mean(np.imag(Mp_I(t, s, g, G_ratio, T, spin_I, DDF_S)))/spin_S.M0

Conc_S = 1e-2 # Concentration of S species in M
M0_S = spindata.magnetization("1H", 9.4, 298, Conc_S)
tau_DDF_S = 1/(M0_S * mu_0 * gamma)
print(f"Magnetization of S spins is {M0_S:.5f} A/m and DDF time is {tau_DDF_S:.5f} s")
spin_S.M0 = M0_S

for idg, g in enumerate(grads):
    for idt, t in enumerate(t2):
        DDF_S = mu_0 * 2 / 3 * Mz_S(t, T, s, g, spin_S)
        signal_x[1, idg, idt] = np.mean(np.real(Mp_I(t, s, g, G_ratio, T, spin_I, DDF_S)))/spin_S.M0
        signal_y[1, idg, idt] = np.mean(np.imag(Mp_I(t, s, g, G_ratio, T, spin_I, DDF_S)))/spin_S.M0

In [ ]:
plt.rcParams.update({'font.size': 12})
fig = plt.figure(figsize=(6, 2*len(grads)), constrained_layout=True)
subfigs = fig.subfigures(1, 2)

# axs[len(grads)-1, 0].set_xlabel("time, s")
# axs[len(grads)-1, 1].set_xlabel("time, s")
titles = [f"[S] = 100 mM",f"[S] = 10 mM"]

for i, subfig in enumerate(subfigs):
    axs = subfig.subplots(len(grads), 1, sharex= True)
    for idg in range(len(grads)):
        if i==0: axs[idg].set_ylabel(r"$|M_{Tr}^{I}(t_{2})|$ / $M_{0}^{S}$")
        axs[idg].set_title(f"G = {grads[idg]*1e+6:.1f} uT/cm",fontsize=14)
        #axs[idg, 0].set_ylim(-4e+3, 4e+3)
        axs[idg].plot(t2, signal_x[i, idg, :], label = "$M_x$")
        axs[idg].plot(t2, signal_y[i, idg, :], label = "$M_y$")
        #axs[idg, 1].set_ylim(-3e+3, 3e+3)
    axs[len(grads)-1].set_xlabel("time, s")
    subfig.suptitle(titles[i])

fig.suptitle(f"DDF signal of Maleate at {P0_I*Conc_I:.2f} M polarization\n" + 
             r'Maleate has T$_{1}$='+f"{spin_I.T1} s, " + r'T$_{2}$='+f"{spin_I.T2} s\n" + 
             r"Substrate has T$_{1}$=" + f"{spin_S.T1} s, " + r'T$_{2}$='+f"{spin_S.T2} s\n"
             
              ,fontsize=16)

plt.show()


### Test inderect DDF with H2O

In [ ]:
B_0 = 9.4 # Field in Teslas
gamma = spindata.gamma("1H")

## Analyte spin S
T1_S = 10  # Longitudinal relaxation time (s)
T2_S = 5  # Transverse relaxation time (s)
omega_offset_S = 2

## Sensor spin I
T1_I = 10  # Longitudinal relaxation time (s)
T2_I = 5  # Transverse relaxation time (s)
omega_offset_I = 50
Conc_I = 110 # Concentration of I species in M
P0_I = spindata.polarization("1H", 9.4, 298)
M0_I = spindata.magnetization("1H", 9.4, 298, Conc_I)
tau_DDF_I = 1/(M0_I * mu_0 * gamma)
print(f"Polatization of I spins is {P0_I:.5f}")
print(f"Magnetization of I spins is {M0_I:.5f} A/m and DDF time is {tau_DDF_I:.5f} s")
spin_I = Spin(M0 = M0_I, omega_offset = omega_offset_I, T1 = T1_I, T2 = T2_I, DDF = True)

# Parameters for geometry
sample_length = 1  # cm
num_slices = 20000  # Number of slices
slice_positions = np.linspace(0, sample_length, num_slices)  # Positions of slices (cm)

In [ ]:
grads = [1e-6, 1e-5, 1e-4]
grad_ratio = 1
t2 = np.linspace(0,50,5000)

T = 0.1

signal_x = np.zeros([2, len(grads), t2.shape[0]])
signal_y = np.zeros([2, len(grads), t2.shape[0]])

Conc_S = 1e-1 # Concentration of S species in M
M0_S = spindata.magnetization("1H", 9.4, 298, Conc_S)
tau_DDF_S = 1/(M0_S * mu_0 * gamma)
print(f"Magnetization of S spins is {M0_S:.5f} A/m and DDF time is {tau_DDF_S:.5f} s")
spin_S = Spin(M0 = M0_S, omega_offset = omega_offset_S, T1 = T1_S, T2 = T2_S, DDF = False)

for idg, g in enumerate(grads):
    for idt, t in enumerate(t2):
        DDF_S = mu_0 * 2 / 3 * Mz_S(t, T, s, g, spin_S)
        signal_x[0, idg, idt] = np.mean(np.real(Mp_I(t, s, g, G_ratio, T, spin_I, DDF_S)))/spin_S.M0
        signal_y[0, idg, idt] = np.mean(np.imag(Mp_I(t, s, g, G_ratio, T, spin_I, DDF_S)))/spin_S.M0

Conc_S = 1e-2 # Concentration of S species in M
M0_S = spindata.magnetization("1H", 9.4, 298, Conc_S)
tau_DDF_S = 1/(M0_S * mu_0 * gamma)
print(f"Magnetization of S spins is {M0_S:.5f} A/m and DDF time is {tau_DDF_S:.5f} s")
spin_S.M0 = M0_S

for idg, g in enumerate(grads):
    for idt, t in enumerate(t2):
        DDF_S = mu_0 * 2 / 3 * Mz_S(t, T, s, g, spin_S)
        signal_x[1, idg, idt] = np.mean(np.real(Mp_I(t, s, g, G_ratio, T, spin_I, DDF_S)))/spin_S.M0
        signal_y[1, idg, idt] = np.mean(np.imag(Mp_I(t, s, g, G_ratio, T, spin_I, DDF_S)))/spin_S.M0

In [ ]:
plt.rcParams.update({'font.size': 12})
fig = plt.figure(figsize=(6, 2*len(grads)), constrained_layout=True)
subfigs = fig.subfigures(1, 2)

# axs[len(grads)-1, 0].set_xlabel("time, s")
# axs[len(grads)-1, 1].set_xlabel("time, s")
titles = [f"[S] = 100 mM",f"[S] = 10 mM"]

for i, subfig in enumerate(subfigs):
    axs = subfig.subplots(len(grads), 1, sharex= True)
    for idg in range(len(grads)):
        if i==0: axs[idg].set_ylabel(r"$|M_{Tr}^{I}(t_{2})|$ / $M_{0}^{I}$")
        axs[idg].set_title(f"G = {grads[idg]*1e+6:.1f} uT/cm",fontsize=14)
        #axs[idg, 0].set_ylim(-4e+3, 4e+3)
        axs[idg].plot(t2, signal_x[i, idg, :], label = "$M_x$")
        axs[idg].plot(t2, signal_y[i, idg, :], label = "$M_y$")
        #axs[idg, 1].set_ylim(-3e+3, 3e+3)
    axs[len(grads)-1].set_xlabel("time, s")
    subfig.suptitle(titles[i])

fig.suptitle(f"DDF signal of H2O\n" + 
             r'H2O has T$_{1}$='+f"{T1_I} s, " + r'T$_{2}$='+f"{T2_I} s\n" + 
             r"Substrate has T$_{1}$=" + f"{T1_S} s, " + r'T$_{2}$='+f"{T2_S} s\n"
             
              ,fontsize=16)

plt.show()

In [ ]:
grads = [1e-6, 1e-5, 1e-4]
grad_ratio = 1
t2 = np.linspace(0,50,5000)

T_1 = 0.1
T_2 = 0.15

signal_x = np.zeros([2, len(grads), t2.shape[0]])
signal_y = np.zeros([2, len(grads), t2.shape[0]])

Conc_S = 1e-1   # Concentration of S species in M
M0_S = spindata.magnetization("1H", 9.4, 298, Conc_S)
tau_DDF_S = 1/(M0_S * mu_0 * gamma)
print(f"Magnetization of S spins is {M0_S:.5f} A/m and DDF time is {tau_DDF_S:.5f} s")
spin_S = Spin(M0 = M0_S, omega_offset = omega_offset_S, T1 = T1_S, T2 = T2_S, DDF = False)

for idg, g in enumerate(grads):
    for idt, t in enumerate(t2):
        DDF_S = mu_0 * 2 / 3 * Mz_S(t, T, s, g, spin_S)
        signal_x[0, idg, idt] = np.mean(np.real(Mp_I(t, s, g, G_ratio, T, spin_I, DDF_S)))/spin_S.M0
        signal_y[0, idg, idt] = np.mean(np.imag(Mp_I(t, s, g, G_ratio, T, spin_I, DDF_S)))/spin_S.M0

for idg, g in enumerate(grads):
    for idt, t in enumerate(t2):
        DDF_S = mu_0 * 2 / 3 * Mz_S(t, T, s, g, spin_S)
        signal_x[1, idg, idt] = np.mean(np.real(Mp_I(t, s, g, G_ratio, 0.5 * T, spin_I, DDF_S)))/spin_S.M0
        signal_y[1, idg, idt] = np.mean(np.imag(Mp_I(t, s, g, G_ratio, 0.5 * T, spin_I, DDF_S)))/spin_S.M0

In [ ]:
plt.rcParams.update({'font.size': 12})
fig = plt.figure(figsize=(6, 2*len(grads)), constrained_layout=True)
subfigs = fig.subfigures(1, 2)

# axs[len(grads)-1, 0].set_xlabel("time, s")
# axs[len(grads)-1, 1].set_xlabel("time, s")
titles = [r"$\tau_{grad_{1}} = \tau_{grad_{2}}$",r"$\tau_{grad_{1}} = 2*\tau_{grad_{2}}$"]

for i, subfig in enumerate(subfigs):
    axs = subfig.subplots(len(grads), 1, sharex= True)
    for idg in range(len(grads)):
        if i==0: axs[idg].set_ylabel(r"$|M_{Tr}^{I}(t_{2})|$ / $M_{0}^{I}$")
        axs[idg].set_title(f"G = {grads[idg]*1e+6:.1f} uT/cm",fontsize=14)
        #axs[idg, 0].set_ylim(-4e+3, 4e+3)
        axs[idg].plot(t2, signal_x[i, idg, :], label = "$M_x$")
        axs[idg].plot(t2, signal_y[i, idg, :], label = "$M_y$")
        axs[idg].set_xlim(0, 3)
        #axs[idg, 1].set_ylim(-3e+3, 3e+3)
    axs[len(grads)-1].set_xlabel("time, s")
    subfig.suptitle(titles[i])

fig.suptitle(f"DDF signal of H2O\n" + 
             r'H2O has T$_{1}$='+f"{T1_I} s, " + r'T$_{2}$='+f"{T2_I} s\n" + 
             r"Substrate has T$_{1}$=" + f"{T1_S} s, " + r'T$_{2}$='+f"{T2_S} s\n"
             
              ,fontsize=16)

plt.show()